# 06C Recovery v2 — Robustly Pack the Existing Focal-Loss Output

This notebook performs **no training**.

## Required Kaggle input

Attach the saved committed output of:

`06C_ResNet50_320_FocalLoss`

Then run this recovery notebook on **CPU**.

This version does **not assume one exact Kaggle mount path**. It searches the attached inputs using several signals:

- `config.json` with experiment ID `L02_ResNet50_320_FocalLoss`
- the experiment directory name
- `summary_metrics.csv`
- `validation_predictions.csv`
- focal-loss-related folder names

It then packages the entire detected experiment output directory into:

`/kaggle/working/L02_ResNet50_320_FocalLoss_All_Outputs.zip`

If the saved 06C version truly contains no persisted output files, the notebook will print the mounted input tree clearly instead of silently guessing.


In [1]:
# ============================================================
# 1. Imports / constants
# ============================================================
from pathlib import Path
import json
import zipfile
import hashlib
import os

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

EXPERIMENT_ID = "L02_ResNet50_320_FocalLoss"
ZIP_PATH = KAGGLE_WORKING / "L02_ResNet50_320_FocalLoss_All_Outputs.zip"

print("Kaggle input exists:", KAGGLE_INPUT.exists())
print("Target experiment:", EXPERIMENT_ID)


Kaggle input exists: True
Target experiment: L02_ResNet50_320_FocalLoss


In [2]:
# ============================================================
# 2. Diagnostic: show mounted Kaggle inputs
# ============================================================
top_level = sorted(KAGGLE_INPUT.iterdir()) if KAGGLE_INPUT.exists() else []

print("Mounted /kaggle/input entries:", len(top_level))
for p in top_level:
    kind = "DIR" if p.is_dir() else "FILE"
    print(f"- [{kind}] {p}")

if not top_level:
    raise FileNotFoundError(
        "No Kaggle inputs are mounted. Add the saved output of "
        "06C_ResNet50_320_FocalLoss using Add Input."
    )


Mounted /kaggle/input entries: 1
- [DIR] /kaggle/input/notebooks


In [3]:
# ============================================================
# 3. Search by config.json first (strongest signal)
# ============================================================
config_matches = []

for cfg_path in KAGGLE_INPUT.rglob("config.json"):
    try:
        cfg = json.loads(cfg_path.read_text())
    except Exception:
        continue

    if cfg.get("experiment_id") == EXPERIMENT_ID:
        config_matches.append(cfg_path)

print("Matching config.json files:", len(config_matches))
for p in config_matches:
    print("-", p)

SOURCE_DIR = None

if config_matches:
    # config.json normally lives directly in the experiment output folder.
    SOURCE_DIR = config_matches[0].parent
    print("\n✅ Found experiment from config.json:")
    print(SOURCE_DIR)


Matching config.json files: 0


In [4]:
# ============================================================
# 4. Fallback searches if config.json is unavailable
# ============================================================
if SOURCE_DIR is None:
    exact_dirs = [
        p for p in KAGGLE_INPUT.rglob(EXPERIMENT_ID)
        if p.is_dir()
    ]

    print("Exact experiment directories:", len(exact_dirs))
    for p in exact_dirs:
        print("-", p)

    if exact_dirs:
        SOURCE_DIR = max(
            exact_dirs,
            key=lambda p: sum(
                f.stat().st_size
                for f in p.rglob("*")
                if f.is_file()
            ),
        )

if SOURCE_DIR is None:
    # Look for key result files and infer their experiment root.
    key_files = []
    for name in [
        "summary_metrics.csv",
        "validation_predictions.csv",
        "training_history.csv",
        "best_model.pth",
    ]:
        key_files.extend(KAGGLE_INPUT.rglob(name))

    # Prefer paths containing focal/L02.
    preferred = [
        p for p in key_files
        if "focal" in str(p).lower()
        or "l02" in str(p).lower()
    ]

    print("Fallback key result files:", len(key_files))
    print("Preferred focal/L02 result files:", len(preferred))

    for p in preferred[:30]:
        print("-", p)

    if preferred:
        p = preferred[0]

        # Walk upward until we find a directory that appears to contain
        # several standard experiment outputs.
        for parent in [p.parent, *p.parents]:
            if parent == KAGGLE_INPUT:
                break

            names = {
                f.name
                for f in parent.rglob("*")
                if f.is_file()
            }

            score = sum(
                x in names for x in [
                    "config.json",
                    "summary_metrics.csv",
                    "training_history.csv",
                    "validation_predictions.csv",
                    "best_model.pth",
                ]
            )

            if score >= 2:
                SOURCE_DIR = parent
                break

if SOURCE_DIR is not None:
    print("\n✅ Selected source directory:")
    print(SOURCE_DIR)


Exact experiment directories: 1
- /kaggle/input/notebooks/kalloldaskushol/06c-resnet50-320-focalloss/PediCGAM/L02_ResNet50_320_FocalLoss

✅ Selected source directory:
/kaggle/input/notebooks/kalloldaskushol/06c-resnet50-320-focalloss/PediCGAM/L02_ResNet50_320_FocalLoss


In [5]:
# ============================================================
# 5. If still missing, print a useful tree and stop
# ============================================================
if SOURCE_DIR is None:
    print("\nNo focal experiment output directory was detected.")
    print("\nDiagnostic file tree (first 250 files):")

    all_files = [
        p for p in KAGGLE_INPUT.rglob("*")
        if p.is_file()
    ]

    for p in all_files[:250]:
        print("-", p)

    print("\nTotal mounted files:", len(all_files))

    raise FileNotFoundError(
        "The attached Kaggle input does not contain persisted 06C result files. "
        "Confirm that you added the SAVED/COMMITTED 06C version whose Output tab "
        "contains the completed run, not the notebook draft."
    )


In [6]:
# ============================================================
# 6. Inspect source and protect restricted image data
# ============================================================
source_files = sorted(
    p for p in SOURCE_DIR.rglob("*")
    if p.is_file()
)

if not source_files:
    raise RuntimeError(
        f"Detected source directory is empty: {SOURCE_DIR}. "
        "The committed 06C version did not persist its run outputs."
    )

total_bytes = sum(p.stat().st_size for p in source_files)

print("Source file count:", len(source_files))
print(f"Source size: {total_bytes/(1024**2):.2f} MB")

for p in source_files:
    rel = p.relative_to(SOURCE_DIR)
    lower = str(rel).lower()

    if lower.endswith(".dcm") or lower.endswith(".dicom"):
        raise RuntimeError(
            f"Restricted DICOM detected: {rel}. "
            "Recovery ZIP creation stopped."
        )

print("✅ No DICOM files found.")


Source file count: 3
Source size: 1.34 MB
✅ No DICOM files found.


In [7]:
# ============================================================
# 7. Create ONE complete ZIP
# ============================================================
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:

    for p in source_files:
        rel = p.relative_to(SOURCE_DIR)

        # Keep a clean experiment folder at ZIP root.
        arcname = Path(EXPERIMENT_ID) / rel
        zf.write(p, arcname=str(arcname))

print("✅ ZIP created")
print("Path:", ZIP_PATH)
print(f"Size: {ZIP_PATH.stat().st_size/(1024**2):.2f} MB")


✅ ZIP created
Path: /kaggle/working/L02_ResNet50_320_FocalLoss_All_Outputs.zip
Size: 1.32 MB


In [8]:
# ============================================================
# 8. Verify ZIP completeness and integrity
# ============================================================
expected = {
    str(Path(EXPERIMENT_ID) / p.relative_to(SOURCE_DIR))
    for p in source_files
}

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    actual = set(zf.namelist())
    corrupt = zf.testzip()

if corrupt is not None:
    raise RuntimeError(f"Corrupt ZIP entry: {corrupt}")

missing = sorted(expected - actual)

if missing:
    raise RuntimeError(
        "ZIP verification failed. Missing entries:\n"
        + "\n".join(missing[:50])
    )

print("ZIP entries:", len(actual))
print("Expected source files:", len(expected))
print("Missing source files:", len(missing))
print("ZIP integrity:", "PASS")

important = [
    "config.json",
    "summary_metrics.csv",
    "training_history.csv",
    "validation_predictions.csv",
    "per_class_metrics.csv",
    "head_middle_tail_metrics.csv",
    "best_model.pth",
]

print("\nImportant output presence:")
for name in important:
    found = any(x.endswith("/" + name) for x in actual)
    print(("✅" if found else "⚠️"), name)


ZIP entries: 3
Expected source files: 3
Missing source files: 0
ZIP integrity: PASS

Important output presence:
⚠️ config.json
⚠️ summary_metrics.csv
⚠️ training_history.csv
⚠️ validation_predictions.csv
⚠️ per_class_metrics.csv
⚠️ head_middle_tail_metrics.csv
⚠️ best_model.pth


In [9]:
# ============================================================
# 9. Final status
# ============================================================
print("\n" + "=" * 72)
print("06C RECOVERY V2 STATUS: PASS")
print("✅ EXISTING OUTPUT RECOVERED — NO RETRAINING")
print("✅ ENTIRE DETECTED EXPERIMENT DIRECTORY PACKED")
print("✅ ONE DOWNLOADABLE ZIP CREATED")
print("✅ NO DICOM DATA INCLUDED")
print("=" * 72)
print("\nDownload:")
print(ZIP_PATH)



06C RECOVERY V2 STATUS: PASS
✅ EXISTING OUTPUT RECOVERED — NO RETRAINING
✅ ENTIRE DETECTED EXPERIMENT DIRECTORY PACKED
✅ ONE DOWNLOADABLE ZIP CREATED
✅ NO DICOM DATA INCLUDED

Download:
/kaggle/working/L02_ResNet50_320_FocalLoss_All_Outputs.zip
